# Chunked reconstruction of Casper (Nikon), as DTU LSF batch jobs

Reconstructs the Nikon scan `Casper_top_3_2` with `chunkct` by submitting DTU LSF batch jobs, then inspects
the result with qim3d.

The scan is a dataset of the 3D Imaging Center at DTU, so you probably cannot run this notebook yourself.
It serves as a worked example for a Nikon scan; for your own data, start from `template.ipynb`.

In [ ]:
from pathlib import Path

import dask.array as da
import matplotlib.pyplot as plt
import numpy as np
import qim3d
import zarr
from cil.utilities.display import show_geometry

from chunkct import NikonZeissTiffProjectionSource
from chunkct.lsf import DTULSFChunkedCT

## Parameters

- `ANGLE_CHUNKS`: number of chunks the projections are split into along the angle axis.
  More chunks means less projection data in memory per FDK call.
- `RECON_CHUNK_SHAPE`: size of the volume block reconstructed by one FDK call (limited by GPU/host memory).
  In LSF mode each block is one GPU job.
- `VOL_CHUNK_SHAPE`: chunk shape of `recon.zarr` on disk, i.e. how the result is stored and read later.
  Must divide `RECON_CHUNK_SHAPE` on every axis.
- `PAD_FACTOR`: horizontal edge-padding of each projection as a fraction of its width (e.g. `0.25`);
  reduces the bright ring at the edge of the field of view. `None` disables it.

In [ ]:
CONFIG_PATH = Path(
    "/dtu/3d-imaging-center/projects/2021_DANFIX_Casper/raw_data_3DIM/"
    "Casper_top_3_2 [2021-03-17 16.54.39]/Casper_top_3_2_recon.xtekct"
)
OUTPUT_DIR = Path("out/casper").resolve()  # absolute, so LSF jobs find it too
JOB_NAME = "CASPER"

ANGLE_CHUNKS = 2
RECON_CHUNK_SHAPE = (500, 500, 500)
VOL_CHUNK_SHAPE = (100, 100, 100)
PAD_FACTOR = 0.25
FILTER = "ram-lak"

## Projection source

`NikonZeissTiffProjectionSource` takes the geometry from the `.xtekct` (via CIL's reader) and reads the
TIFF projections from the same folder. Raw counts are normalised by the `WhiteLevel` from the `.xtekct`.

Note: the folder contains 1572 TIFFs but the scan has 1571 projections; the last TIFF repeats the 0°
angle and is not used.

In [ ]:
source = NikonZeissTiffProjectionSource(config_path=CONFIG_PATH)
ag, ig = source.get_geometries()

print(f"projections (angles, rows, cols): {ag.shape}")
print(f"volume (z, y, x):                 {ig.shape}")
print(f"voxel size:                       {ig.voxel_size_x:.4f}")
print(f"WhiteLevel (normalisation):       {source._normalization_constant}")
print(f"TIFF files found:                 {len(source.paths)}")

In [ ]:
show_geometry(ag);

In [ ]:
# One raw projection (uint16 counts) and its transmission after normalisation
raw = source.read_chunk(0, 1)
transmission = source.process_transmission_chunk(raw.astype(np.float32))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, img, title in zip(
    axes, (raw[0], transmission[0]), ("raw counts", "transmission"), strict=True
):
    im = ax.imshow(img, cmap="gray")
    ax.set_title(title)
    fig.colorbar(im, ax=ax)

## Submit DTU LSF batch jobs

One CPU job writes the projections into a compressed `projections.zarr`; then one GPU job per volume block
reconstructs it (each waits for the write job) and writes its block into `recon.zarr`. Jobs run with this
notebook's Python interpreter; job scripts and logs go to `OUTPUT_DIR/bjobs/run_<timestamp>/`.

Start with `dry_run=True`: this only writes the job scripts so you can inspect them in the run directory.
Set it to `False` to actually submit.

In [ ]:
reconstructor = DTULSFChunkedCT(
    projection_source=source,
    output_dir=OUTPUT_DIR,
    angle_chunks=ANGLE_CHUNKS,
    recon_chunk_shape=RECON_CHUNK_SHAPE,
    vol_chunk_shape=VOL_CHUNK_SHAPE,
    pad_factor=PAD_FACTOR,
    filter=FILTER,
)

In [ ]:
run_dir = reconstructor.submit_jobs(job_name=JOB_NAME, dry_run=False)

In [ ]:
reconstructor.check_jobs(JOB_NAME)

In [ ]:
reconstructor.summarize_run(run_dir)

## Look at the result

`recon.zarr` is opened lazily with dask; qim3d's slicer only loads the slice being shown. The colour range
is fixed from percentiles of the central slice so the whole 4 GB volume is not scanned.

In [ ]:
recon = da.from_zarr(zarr.open_array(OUTPUT_DIR / "recon.zarr", mode="r"))
recon

In [ ]:
center_slice = recon[recon.shape[0] // 2].compute()
vmin, vmax = np.percentile(center_slice, [0.5, 99.5])
print(f"display range: {vmin:.4f} .. {vmax:.4f}")

In [ ]:
qim3d.viz.slicer(recon, slice_axis=0, colormap="gray", min_value=vmin, max_value=vmax)